In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, roc_auc_score, 
                             average_precision_score, confusion_matrix)
from sklearn.preprocessing import label_binarize
import joblib

df = pd.read_csv('../data/processed/synthetic_dataset3.csv')
X = df.drop(columns=['diagnosis'])
y = df['diagnosis'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM':           SVC(kernel='rbf', probability=True, random_state=42),
    'MLP':           MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42),
}

In [8]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    
    report = classification_report(y_test, y_pred, output_dict=True)
    auc_roc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
    y_bin = label_binarize(y_test, classes=[1, 2, 3])
    auc_pr = average_precision_score(y_bin, y_proba, average='macro')
    
    results.append({
        'Model':         name,
        'Accuracy':      round(report['accuracy'], 3),
        'Precision (macro)': round(report['macro avg']['precision'], 3),
        'Recall (macro)':    round(report['macro avg']['recall'], 3),
        'F1 (macro)':        round(report['macro avg']['f1-score'], 3),
        'AUC-ROC':       round(auc_roc, 3),
        'AUC-PR':        round(auc_pr, 3),
    })
    
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=['Healthy','Benign','Cancer']))
    print(f"  AUC-ROC : {auc_roc:.3f}")
    print(f"  AUC-PR  : {auc_pr:.3f}")



  Random Forest
              precision    recall  f1-score   support

     Healthy       0.93      0.86      0.89        58
      Benign       0.65      0.84      0.73        37
      Cancer       0.89      0.64      0.74        25

    accuracy                           0.81       120
   macro avg       0.82      0.78      0.79       120
weighted avg       0.83      0.81      0.81       120

  AUC-ROC : 0.939
  AUC-PR  : 0.865

  SVM
              precision    recall  f1-score   support

     Healthy       0.92      0.83      0.87        58
      Benign       0.53      0.73      0.61        37
      Cancer       0.65      0.44      0.52        25

    accuracy                           0.72       120
   macro avg       0.70      0.67      0.67       120
weighted avg       0.74      0.72      0.72       120

  AUC-ROC : 0.885
  AUC-PR  : 0.735

  MLP
              precision    recall  f1-score   support

     Healthy       0.89      0.72      0.80        58
      Benign       0.48   

In [10]:
summary = pd.DataFrame(results).set_index('Model')
print(summary.to_string())

               Accuracy  Precision (macro)  Recall (macro)  F1 (macro)  AUC-ROC  AUC-PR
Model                                                                                  
Random Forest     0.808              0.820           0.780       0.789    0.939   0.865
SVM               0.717              0.700           0.666       0.670    0.885   0.735
MLP               0.642              0.617           0.604       0.603    0.803   0.626


In [12]:
rf_model = models['Random Forest']
joblib.dump(rf_model,'../models/ds3/ds3_random_forest.pkl')

svm_model = models['SVM']
joblib.dump(svm_model,'../models/ds3/ds3_svm.pkl')

mlp_model = models['MLP']
joblib.dump(mlp_model,'../models/ds3/ds3_mlp.pkl')

np.save('../models/ds3/X_train.npy', X_train)
np.save('../models/ds3/X_test.npy', X_test)
np.save('../models/ds3/y_test.npy', y_test)
pd.Series(X.columns.tolist()).to_csv('../models/ds3/feature_names.csv', index=False)